# SQL and real emissions data

Every dataset you have used so far in dewlab has probably come in as a
`pandas` DataFrame. This one starts the same way — but then it goes into
a real SQL database, and you query it with SQL instead of pandas
methods. Same data, a different tool for asking it questions.

The data itself is real: national CO₂ emissions from
[Our World in Data](https://ourworldindata.org/co2-and-greenhouse-gas-emissions),
who compile it from the Global Carbon Project and other sources and
publish it under a Creative Commons BY licence. It is shipped with
dewlab as `data/co2-emissions.csv` — one row per country per year,
1950–2023.

Run the next cell before reading on. Seeing the table appear is worth
more than a description of it.

In [ ]:
df = await load_csv("co2-emissions.csv")
show_table(df.head())

Nine columns: a country and its three-letter code, a year, population
and GDP, total CO₂ emissions for that year (`co2`, in million tonnes),
`co2_per_capita`, and how much of that CO₂ came from coal, oil, and gas
burned, plus cement production. Two more — `methane` and
`nitrous_oxide` — are other greenhouse gases, and `temperature_change_from_co2`
estimates that country's own contribution to global warming so far, in
degrees.

## Into SQLite

`sqlite3` is a real relational database — the same kind of thing behind
plenty of real applications — built into Python, so there's nothing to
install. A DataFrame becomes a table with one line.

In [ ]:
import sqlite3

conn = sqlite3.connect(":memory:")   # a database that lives only in memory, for this session
df.to_sql("emissions", conn, index=False)

`:memory:` means the database exists only for as long as this cell's
connection stays open — nothing is written to a file. That's fine here;
if you wanted the database to survive a reload, you'd connect to a real
filename instead (`sqlite3.connect("emissions.db")`), and it would show
up in the Files pane.

## Asking questions in SQL instead of pandas

dewlab's `run_query()` runs a SQL string against a connection and shows
the result as a table — the SQL equivalent of `show_table()`. Which
eight countries emitted the most CO₂ in the most recent year on record?

In [ ]:
run_query(conn, """
    select country, co2, co2_per_capita
    from emissions
    where year = 2023
    order by co2 desc
    limit 8
""")

`order by co2 desc` sorts biggest first; `limit 8` keeps only the top
eight. If you have used pandas' `.sort_values()` and `.head()`, this is
the same two ideas, spelled differently.

Look at the two columns side by side. Total emissions and per-capita
emissions tell different stories about the same countries — a country
can lead one ranking and barely register on the other. Try changing
`co2 desc` to `co2_per_capita desc` in the cell above and re-running it.
What changes about *which* countries appear?

In [ ]:
# Your turn: which countries lead the per-capita ranking instead?
# Copy the query above, swap the order by column, and run it.


## Grouping and aggregating

SQL's `group by` collapses many rows into one per group — here, one row
per country — while an aggregate function like `avg()` summarises what
was in each group. This finds each country's average per-capita
emissions over the last decade of the data:

In [ ]:
run_query(conn, """
    select country, round(avg(co2_per_capita), 2) as avg_per_capita
    from emissions
    where year between 2014 and 2023
    group by country
    order by avg_per_capita desc
    limit 8
""")

Notice who shows up here that didn't show up in the single-year ranking
above — small, oil- and gas-producing states, mostly, whose *average*
holds steady near the top even though any single year bounces around.
`group by` is what makes that kind of "on average, over time" question
answerable in one query rather than a loop.

## One country over time

SQL isn't only for ranking everything against everything else — a plain
`where` narrows to one country, and reading the rows in order tells its
own story.

In [ ]:
ireland = run_query(conn, """
    select year, co2, co2_per_capita
    from emissions
    where country = 'Ireland'
    order by year
""", max_rows=5)

`run_query()` always hands back the full result as a DataFrame too —
`max_rows=5` only limits what gets *displayed*, not what `ireland` holds
— so you can immediately pass it to `matplotlib`, exactly as if it had
come from `load_csv()` in the first place.

In [ ]:
import matplotlib.pyplot as plt

plt.plot(ireland["year"], ireland["co2_per_capita"])
plt.title("Ireland's CO2 emissions per person, 1950–2023")
plt.xlabel("Year")
plt.ylabel("Tonnes of CO2 per person")
plt.show()

Pick a country of your own and adapt the cell above — just change the
name in `where country = '...'`. A shape worth noticing when you do: a
climb through the twentieth century, then in many industrialised
countries a peak somewhere after 1990 and a decline since. Does the
country you picked follow that shape, or something else entirely? Either
answer is a real finding about a real place — that's what happens when
you can just ask the data.

## Your turn

Using `run_query()`, find the five countries with the lowest
`temperature_change_from_co2` among countries where `co2_per_capita` in
2023 was above 5 — i.e., countries that burn a fair amount of fuel per
person today but have contributed comparatively little to warming so
far. (`hint:` a `where` clause can combine two conditions with `and`,
and you will need both a `year = 2023` filter and one on
`co2_per_capita` to identify *which* countries qualify, before ordering
by their long-run `temperature_change_from_co2`.)

In [ ]:
# Your turn — five countries, high recent per-capita emissions,
# low cumulative warming contribution. Start from the query pattern above.


## Where this can go next

Everything above ran one query at a time against one table. Real
databases usually have several tables that relate to each other — a
`students` table and a `grades` table, sharing a student id — joined
together with SQL's `join`. `run_query()` handles a join exactly the
same way: write the SQL, get a table back. If you build your own
multi-table database — in the Files pane, with `sqlite3.connect("your-name.db")`
instead of `:memory:` — it will still be there the next time you open
Mini IDE.

**Source**: Hannah Ritchie, Pablo Rosado and Max Roser (2023) — ["CO₂ and Greenhouse Gas Emissions"](https://ourworldindata.org/co2-and-greenhouse-gas-emissions).
Published online at OurWorldInData.org. Licensed
[CC BY 4.0](https://creativecommons.org/licenses/by/4.0/).